In [8]:
%pip install "yt-dlp[default,curl-cffi]"


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
URL = ["https://youtu.be/YSJY3DvnybE?si=Cha2S3kOlVUWjxwK"]

In [10]:
from pathlib import Path

from yt_dlp import YoutubeDL

OUTPUT_DIR = Path("Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def download_with_subs(urls: list[str]) -> None:
    """Tải subtitle tiếng Việt.
    - Thử 'vi' trước.
    - Nếu không có, thử 'vi-en' (auto-translate YouTube).
    - Đổi tên file về <stem>.vi.vtt để các cell sau dùng thống nhất.
    """
    for lang in ("vi", "vi-en"):
        opts = {
            "paths": {"home": str(OUTPUT_DIR)},
            "writesubtitles": True,
            "writeautomaticsub": True,
            "subtitleslangs": [lang],
            "skip_download": False,
            "ignoreerrors": True,
        }
        print(f"Thử tải subtitle [{lang}]…")
        with YoutubeDL(opts) as ydl:
            ydl.download(urls)

        # Đổi tên: bất kỳ *.vi-en.vtt hoặc *.vi.vtt → cùng tên nhưng suffix chuẩn .vi.vtt
        renamed = 0
        for f in list(OUTPUT_DIR.glob("*.vtt")):
            stem = f.stem  # e.g. "Title [id].vi-en" hoặc "Title [id].vi"
            if stem.endswith(".vi-en"):
                new_name = stem[: -len(".vi-en")] + ".vi.vtt"
                target = f.parent / new_name
                if not target.exists():
                    f.rename(target)
                    print(f"  Renamed: {f.name} → {target.name}")
                renamed += 1

        # Kiểm tra có file .vi.vtt mới nào không
        vi_vtts = list(OUTPUT_DIR.glob("*.vi.vtt"))
        if vi_vtts:
            print(f"OK: {len(vi_vtts)} file .vi.vtt trong Output/")
            return

        print(f"Không tìm thấy subtitle [{lang}], thử ngôn ngữ tiếp theo…")

    print("Không tải được subtitle vi hoặc vi-en cho bất kỳ URL nào.")


# Bỏ qua nếu tất cả URL đã có subtitle
existing = sorted(OUTPUT_DIR.glob("*.vi.vtt"))
if existing:
    print(f"Đã có {len(existing)} file .vi.vtt — bỏ qua tải subtitle.")
    for f in existing:
        print(f"  {f.name}")
else:
    download_with_subs(URL)


Thử tải subtitle [vi]…
[youtube] Extracting URL: https://youtu.be/YSJY3DvnybE?si=Cha2S3kOlVUWjxwK
[youtube] YSJY3DvnybE: Downloading webpage
[youtube] YSJY3DvnybE: Downloading android vr player API JSON
[youtube] YSJY3DvnybE: Downloading player 8e54e4ea-main
[youtube] [jsc:deno] Solving JS challenges using deno
[youtube] YSJY3DvnybE: Downloading m3u8 information
[info] YSJY3DvnybE: Downloading subtitles: vi
[info] YSJY3DvnybE: Downloading 1 format(s): 401+251
[info] Writing video subtitles to: Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].vi.vtt


[download] Destination: Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].f401.mp4
[download] 100% of  803.34MiB in 00:02:07 at 6.29MiB/s     
[download] Destination: Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].f251.webm
[download] 100% of   27.87MiB in 00:00:04 at 5.96MiB/s     
[Merger] Merging formats into "Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].webm"
Deleting original file Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].f401.mp4 (pass -k to keep)
Deleting original file Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].f251.webm (pass -k to keep)
Không tìm thấy subtitle [vi], thử ngôn ngữ tiếp theo…
Thử tải subtitle [vi-en]…
[youtube] Extracting URL: https://youtu.be/YSJY3DvnybE?si=Cha2S3kOlVUWjxwK
[youtube] YSJY3DvnybE: Downloading webpage
[youtube] YSJY3DvnybE: Downloading android vr player API JSON
[youtube] YSJY3DvnybE: Downloading player 8e54e4ea-main
[youtube] [jsc:deno] Solving 

[download] Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].webm has already been downloaded
Không tìm thấy subtitle [vi-en], thử ngôn ngữ tiếp theo…
Không tải được subtitle vi hoặc vi-en cho bất kỳ URL nào.


In [11]:
%pip install vieneu --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/metal/


Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/metal/

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import re
from pathlib import Path

OUTPUT_DIR = Path("Output")

TS_LINE = re.compile(
    r"^\s*(\d{2}:\d{2}:\d{2}[.,]\d{3})\s+-->\s+(\d{2}:\d{2}:\d{2}[.,]\d{3})"
)
KARAOKE = re.compile(r"<(\d{2}:\d{2}:\d{2})[.,](\d{3})><c>(.*?)</c>", re.DOTALL)
# Tất cả dấu câu kết thúc; is_sentence_boundary() quyết định . có phải ranh giới không
PUNCT_END = re.compile(r"[.!?。…]+\s*")


def is_sentence_boundary(text: str, pos: int) -> bool:
    """Trả về True nếu ký tự tại pos là ranh giới kết thúc câu.
    Dấu . KHÔNG là ranh giới khi kẹp giữa hai chữ số (10.000, 3.14)
    hoặc kẹp giữa hai chữ cái (U.S.A, v.v.).
    """
    if text[pos] != ".":
        return True
    prev_ch = text[pos - 1] if pos > 0 else ""
    next_ch = text[pos + 1] if pos + 1 < len(text) else ""
    if prev_ch.isdigit() and next_ch.isdigit():
        return False
    if prev_ch.isalpha() and next_ch.isalpha():
        return False
    return True
# Dấu người đọc YouTube dạng "- [Tên] " hoặc "- [Tên]" ở đầu dòng
SPEAKER_LABEL = re.compile(r"^(?:-\s*)?\[[^\]]*\]\s*")
# Chú thích âm thanh/hành động trong ngoặc đơn, kể cả kéo dài nhiều cue
PAREN_NOTE = re.compile(r"\([^)]*\)")


def clean_text_line(s: str) -> str:
    """Bỏ prefix người đọc [Name], chú thích (âm thanh), khoảng trắng thừa."""
    s = SPEAKER_LABEL.sub("", s)
    s = PAREN_NOTE.sub("", s)
    # Dọn dấu câu/khoảng trắng thừa sát nhau sau khi xóa ngoặc
    s = re.sub(r"[,;:\s]+$", "", s)   # đuôi thừa
    s = re.sub(r"^[,;:\s]+", "", s)   # đầu thừa
    return re.sub(r"\s+", " ", s).strip()


def parse_vtt_ts_fragment(ts: str) -> float:
    ts = ts.strip().replace(",", ".")
    h, m, s = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


def iter_raw_vtt_cues(path: Path):
    lines = path.read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n").split("\n")
    i, n = 0, len(lines)
    while i < n:
        m = TS_LINE.match(lines[i])
        if not m:
            i += 1
            continue
        t0 = parse_vtt_ts_fragment(m.group(1))
        t1 = parse_vtt_ts_fragment(m.group(2))
        i += 1
        while i < n and not lines[i].strip():
            i += 1
        payload: list[str] = []
        while i < n and lines[i].strip() and not TS_LINE.match(lines[i]):
            if lines[i].strip().upper().startswith("NOTE"):
                break
            payload.append(lines[i].strip())
            i += 1
        while i < n and not lines[i].strip():
            i += 1
        yield t0, t1, payload


def karaoke_blob_from_payload(payload: list[str]) -> str:
    """Chỉ giữ dòng có tag thời gian karaoke; bỏ dòng rollup trùng (YouTube).
    Loại bỏ prefix người đọc dạng - [Name] trước khi ghép."""
    cleaned = [clean_text_line(ln) for ln in payload]
    tagged = [ln for ln in cleaned if re.search(r"<\d{2}:\d{2}:\d{2}", ln)]
    if tagged:
        return " ".join(tagged)
    return " ".join(cleaned)


def parse_karaoke_time(m: re.Match) -> float:
    return parse_vtt_ts_fragment(f"{m.group(1)}.{m.group(2)}")


def fragments_from_karaoke(cue_t0: float, cue_t1: float, s: str) -> list[tuple[float, float, str]]:
    """(t_start, t_end, text) cho từng mảnh; thời gian lấy từ tag + biên cue."""
    matches = list(KARAOKE.finditer(s))
    if not matches:
        t = re.sub(r"<[^>]+>", "", s)
        t = re.sub(r"\s+", " ", t).strip()
        if not t:
            return []
        return [(cue_t0, cue_t1, t)]

    out: list[tuple[float, float, str]] = []
    pre = s[: matches[0].start()]
    pre = re.sub(r"<[^>]+>", "", pre)
    pre = re.sub(r"\s+", " ", pre).strip()
    if pre:
        t_first = parse_karaoke_time(matches[0])
        out.append((cue_t0, t_first, pre))

    for j, m in enumerate(matches):
        ts = parse_karaoke_time(m)
        te = parse_karaoke_time(matches[j + 1]) if j + 1 < len(matches) else cue_t1
        chunk = m.group(3).replace("\n", " ")
        out.append((ts, te, chunk))

    post = s[matches[-1].end() :]
    post = re.sub(r"<[^>]+>", "", post)
    post = re.sub(r"\s+", " ", post).strip()
    if post:
        t_last = parse_karaoke_time(matches[-1])
        out.append((t_last, cue_t1, post))
    return out


def build_timed_text_stream(path: Path) -> tuple[str, list[float]]:
    """Toàn bộ phụ đề: một chuỗi + thời gian từng ký tự (nội suy trong mảnh)."""
    frags: list[tuple[float, float, str]] = []
    for cue_t0, cue_t1, payload in iter_raw_vtt_cues(path):
        if cue_t1 - cue_t0 < 0.05:
            continue
        blob = karaoke_blob_from_payload(payload)
        frags.extend(fragments_from_karaoke(cue_t0, cue_t1, blob))

    parts: list[str] = []
    times: list[float] = []
    pos = 0
    for ts, te, raw in frags:
        piece = raw.strip()
        if not piece:
            continue
        if parts:
            parts.append(" ")
            nsp = 1
            for k in range(nsp):
                times.append(ts)
            pos += nsp
        n = len(piece)
        if n == 1:
            times.append((ts + te) / 2)
        else:
            for k in range(n):
                times.append(ts + (te - ts) * (k / (n - 1)))
        parts.append(piece)
        pos += n
    full_text = "".join(parts)
    # Xóa nốt các (chú thích) còn sót kéo dài qua nhiều fragment
    # Cần đồng bộ xóa cả ký tự lẫn times tương ứng
    cleaned_chars: list[str] = []
    cleaned_times: list[float] = []
    inside = 0
    for ch, t in zip(full_text, times):
        if ch == "(":
            inside += 1
        if inside == 0:
            cleaned_chars.append(ch)
            cleaned_times.append(t)
        if ch == ")":
            inside = max(inside - 1, 0)
    # Chuẩn hóa khoảng trắng thừa sau khi xóa
    clean_joined = re.sub(r" {2,}", " ", "".join(cleaned_chars)).strip()
    final_times: list[float] = []
    j = 0
    for ch in clean_joined:
        while j < len(cleaned_chars) and cleaned_chars[j] != ch:
            j += 1
        if j < len(cleaned_times):
            final_times.append(cleaned_times[j])
            j += 1
        else:
            final_times.append(cleaned_times[-1] if cleaned_times else 0.0)
    return clean_joined, final_times


def split_sentences_spans(text: str) -> list[tuple[int, int, str]]:
    """Câu hoàn chỉnh dựa trên dấu câu.
    Dấu . không ngắt khi kẹp giữa hai chữ số hoặc hai chữ cái (số/viết tắt).
    """
    spans: list[tuple[int, int, str]] = []
    start = 0
    for m in PUNCT_END.finditer(text):
        if not is_sentence_boundary(text, m.start()):
            continue
        end = m.end()
        sent = text[start:end].strip()
        if sent:
            spans.append((start, end, sent))
        start = end
    if start < len(text):
        tail = text[start:].strip()
        if tail:
            spans.append((start, len(text), tail))
    return spans


def char_time_at(times: list[float], text: str, idx: int) -> float:
    if not times or idx >= len(text):
        return times[-1] if times else 0.0
    if idx < 0:
        return times[0]
    return times[idx]


def sentences_with_timing(path: Path) -> list[tuple[float, float, str]]:
    text, times = build_timed_text_stream(path)
    if not text or len(times) != len(text):
        raise RuntimeError("timed text length mismatch")
    out: list[tuple[float, float, str]] = []
    for s0, s1, sent in split_sentences_spans(text):
        lo = min(s0, len(text) - 1)
        hi = min(s1 - 1, len(text) - 1)
        if hi < lo:
            continue
        t0 = char_time_at(times, text, lo)
        t1 = char_time_at(times, text, hi)
        if t1 < t0:
            t0, t1 = t1, t0
        if t1 - t0 < 0.02:
            t1 = t0 + 0.02
        out.append((t0, t1, sent))
    return out


def fmt_webvtt_ts(sec: float) -> str:
    ms = int(round(sec * 1000))
    s, ms = divmod(ms, 1000)
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


def write_normalized_vtt(src: Path, cues: list[tuple[float, float, str]]) -> Path:
    out_path = src.with_name(f"{src.stem}_normalized.vtt")
    lines = [
        "WEBVTT",
        "Language: vi",
        "",
        "NOTE",
        "Chuẩn hóa: tách câu (. ! ? 。 …), thời gian từ tag karaoke YouTube.",
        "",
    ]
    for t0, t1, text in cues:
        lines.append(f"{fmt_webvtt_ts(t0)} --> {fmt_webvtt_ts(t1)}")
        lines.append(text)
        lines.append("")
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return out_path


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
vtt_inputs = sorted(OUTPUT_DIR.glob("*.vi.vtt"))
if not vtt_inputs:
    print("No Output/*.vi.vtt — run the yt-dlp download cell first.")
for vtt in vtt_inputs:
    out_path = vtt.with_name(f"{vtt.stem}_normalized.vtt")
    if out_path.exists():
        print(f"Skip (đã có): {out_path.name}")
        continue
    try:
        cues = sentences_with_timing(vtt)
    except Exception as e:
        print(f"Skip {vtt.name}: {e}")
        continue
    if not cues:
        print(f"Skip (no cues): {vtt.name}")
        continue
    out_path = write_normalized_vtt(vtt, cues)
    print(f"{vtt.name} -> {out_path.name} ({len(cues)} câu)")


The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].vi.vtt -> The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].vi_normalized.vtt (386 câu)


In [20]:
import re
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from vieneu import Vieneu

OUTPUT_DIR = Path("Output")

TS_LINE = re.compile(
    r"^\s*(\d{2}:\d{2}:\d{2}[.,]\d{3})\s+-->\s+(\d{2}:\d{2}:\d{2}[.,]\d{3})"
)

# Chỉ co ngắn (rate > 1). RATE_MAX giới hạn mức co tối đa để tránh méo.
RATE_MAX = 2.8

# n_fft nhỏ hơn → ít smearing → giọng nói nghe ít "vang/hầm" hơn
STRETCH_N_FFT = 512
STRETCH_HOP = STRETCH_N_FFT // 4


def parse_vtt_ts_fragment(ts: str) -> float:
    ts = ts.strip().replace(",", ".")
    h, m, s = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


def parse_normalized_cues(path: Path) -> list[tuple[float, float, str]]:
    lines = path.read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n").split("\n")
    i, n = 0, len(lines)
    out: list[tuple[float, float, str]] = []
    while i < n:
        m = TS_LINE.match(lines[i])
        if not m:
            i += 1
            continue
        t0 = parse_vtt_ts_fragment(m.group(1))
        t1 = parse_vtt_ts_fragment(m.group(2))
        i += 1
        while i < n and not lines[i].strip():
            i += 1
        if i >= n or TS_LINE.match(lines[i]):
            continue
        text_lines: list[str] = []
        while i < n and lines[i].strip() and not TS_LINE.match(lines[i]):
            ls = lines[i].strip()
            if ls.upper().startswith("NOTE"):
                i += 1
                while i < n and lines[i].strip():
                    i += 1
                while i < n and not lines[i].strip():
                    i += 1
                continue
            text_lines.append(ls)
            i += 1
        text = " ".join(text_lines).strip()
        while i < n and not lines[i].strip():
            i += 1
        if text:
            out.append((t0, t1, text))
    return out


def ensure_mono_float(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1).astype(np.float32)
    return y


def stretch_to_duration(y: np.ndarray, sr: int, dur_target: float) -> np.ndarray:
    """Chỉ co ngắn âm thanh khi dài hơn dur_target (rate > 1), giữ nguyên nếu ngắn hơn.

    - Không kéo dãn (tránh artifact phase vocoder khi giọng đọc vừa khít / nhanh hơn).
    - Vẫn dùng n_fft nhỏ và peak-normalize khi co ngắn.
    """
    dur_target = max(dur_target, 0.02)
    dur_actual = len(y) / sr
    if dur_actual < 1e-6:
        return np.zeros(0, dtype=np.float32)
    rate = dur_actual / dur_target
    if rate <= 1.0:
        # Âm thanh ngắn hơn / bằng khung → trả về nguyên, không pad
        return np.asarray(y, dtype=np.float32)
    rate = float(np.clip(rate, 1.0, RATE_MAX))
    n_target = int(round(dur_target * sr))
    y_st = librosa.effects.time_stretch(
        np.asarray(y, dtype=np.float32),
        rate=rate,
        n_fft=STRETCH_N_FFT,
        hop_length=STRETCH_HOP,
    )
    in_peak = float(np.max(np.abs(y)))
    out_peak = float(np.max(np.abs(y_st)))
    if out_peak > 1e-8 and in_peak > 1e-8:
        y_st = y_st * (in_peak / out_peak)
    if len(y_st) > n_target:
        y_st = y_st[:n_target]
    elif len(y_st) < n_target:
        y_st = np.pad(y_st, (0, n_target - len(y_st)))
    return np.clip(y_st, -1.0, 1.0).astype(np.float32)


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tts = Vieneu()
sr = int(tts.sample_rate)

norm_files = sorted(OUTPUT_DIR.glob("*_normalized.vtt"))
if not norm_files:
    print("No Output/*_normalized.vtt — chạy cell chuẩn hóa subtitle trước.")

for norm in norm_files:
    cues = parse_normalized_cues(norm)
    if not cues:
        print(f"Skip (no cues): {norm.name}")
        continue
    base = norm.stem.removesuffix("_normalized")
    root = base.removesuffix(".vi") if base.endswith(".vi") else base
    out_wav = OUTPUT_DIR / f"{root}_vi_timeline.wav"
    total_end = max(t1 for _, t1, _ in cues) + 0.5
    n_mix = int(round(total_end * sr))
    mix = np.zeros(n_mix, dtype=np.float32)

    if out_wav.exists():
        print(f"Skip (đã có): {out_wav.name}")
        continue
    print(f"{norm.name}: {len(cues)} câu -> 1 file (time-stretch khớp timeline, giữ pitch) -> {out_wav.name}")
    for idx, (t0, t1, text) in enumerate(cues, start=1):
        if not text.strip():
            continue
        dur = max(t1 - t0, 0.02)
        raw = tts.infer(text=text)
        y = ensure_mono_float(raw)
        seg = stretch_to_duration(y, sr, dur)
        start = int(round(t0 * sr))
        end = start + len(seg)
        if end > len(mix):
            mix = np.pad(mix, (0, end - len(mix)))
        mix[start:end] += seg
        if idx % 50 == 0 or idx == len(cues):
            print(f"  {idx}/{len(cues)}")

    peak = float(np.max(np.abs(mix))) if mix.size else 0.0
    if peak > 1.0:
        mix = np.clip(mix * (0.98 / peak), -1.0, 1.0)

    sf.write(str(out_wav), mix, sr, subtype="PCM_16")
    print(f"Done: {out_wav} ({mix.shape[0] / sr:.1f}s, sr={sr})")


llama_context: n_ctx_per_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64  

The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE].vi_normalized.vtt: 386 câu -> 1 file (time-stretch khớp timeline, giữ pitch) -> The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE]_vi_timeline.wav
  50/386
  100/386
  150/386
  200/386
  250/386
  300/386
  350/386
  386/386
Done: Output/The Secret Spy Tech Inside Every Credit Card [YSJY3DvnybE]_vi_timeline.wav (1712.4s, sr=24000)


In [ ]:
import shutil
import subprocess
from pathlib import Path

OUTPUT_DIR = Path("Output")
ORIG_VOL = 0.20   # âm lượng tiếng gốc (0.0–1.0)

# codec âm thanh + sample rate theo container đầu ra
CONTAINER_AUDIO = {
    ".mp4":  ("aac",     "192k", None),
    ".webm": ("libopus", "128k", "48000"),
    ".mkv":  ("aac",     "192k", None),
}


def find_bundles() -> list[tuple[Path, Path, Path]]:
    """(video_gốc, wav_vi, video_output) chưa xử lý."""
    bundles = []
    for norm in sorted(OUTPUT_DIR.glob("*_normalized.vtt")):
        base = norm.stem.removesuffix("_normalized")
        root = base.removesuffix(".vi") if base.endswith(".vi") else base
        wav = OUTPUT_DIR / f"{root}_vi_timeline.wav"
        if not wav.exists():
            wav = OUTPUT_DIR / f"{base}_vi_timeline.wav"
        if not wav.exists():
            continue
        for ext in (".mp4", ".webm", ".mkv"):
            vid = OUTPUT_DIR / f"{root}{ext}"
            if vid.exists():
                out = OUTPUT_DIR / f"{root}_vi{ext}"   # giữ cùng container
                bundles.append((vid, wav, out))
                break
    return bundles


def probe_has_audio(path: Path) -> bool:
    r = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a:0",
         "-show_entries", "stream=codec_name", "-of", "default=nw=1", str(path)],
        capture_output=True, text=True,
    )
    return bool(r.stdout.strip())


if not shutil.which("ffmpeg"):
    print("ffmpeg không tìm thấy — cài bằng: brew install ffmpeg")
else:
    bundles = find_bundles()
    if not bundles:
        print(
            "Không tìm thấy bộ đủ: video (.mp4/.webm/.mkv) + *_vi_timeline.wav "
            "trong Output/. Chạy cell TTS trước."
        )
    for vid, wav, out in bundles:
        if out.exists():
            print(f"Skip (đã có): {out.name}")
            continue

        ext = vid.suffix.lower()
        acodec, abitrate, asr = CONTAINER_AUDIO.get(ext, ("aac", "192k", None))
        has_orig = probe_has_audio(vid)
        print(f"Ghép: {vid.name}")
        print(f"  + {wav.name}")
        print(f"  -> {out.name}  (acodec={acodec}, orig_vol={ORIG_VOL})")

        # Resample WAV tới sample rate cần thiết (opus yêu cầu 48000)
        tts_filter = f"[1:a]aresample=async=1{f':out_sample_rate={asr}' if asr else ''}[a_tts]"

        if has_orig:
            filter_complex = (
                f"[0:a]volume={ORIG_VOL}[a_orig];"
                f"{tts_filter};"
                "[a_orig][a_tts]amix=inputs=2:duration=first:dropout_transition=0[a_out]"
            )
        else:
            filter_complex = f"{tts_filter.replace('[a_tts]', '[a_out]')}"

        cmd = [
            "ffmpeg", "-y",
            "-i", str(vid),
            "-i", str(wav),
            "-filter_complex", filter_complex,
            "-map", "0:v",
            "-map", "[a_out]",
            "-c:v", "copy",
            "-c:a", acodec, "-b:a", abitrate,
            str(out),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            size_mb = out.stat().st_size / 1024 / 1024
            print(f"  Done: {out.name} ({size_mb:.1f} MB)")
        else:
            print(f"  Lỗi ffmpeg:\n{result.stderr[-1200:]}")
